# Électricité mondiale : trajectoires, mix et pression numérique

Données :
- `../data/preproc/world_total_electricity_generation.xlsx` (historical/current/stated/netzero).
- `../data/preproc/total_power_generation.xlsx` (mix par techno).
- `../data/preproc/data_center_electricity_demand_TWh.xlsx` (IA/crypto) pour contextualiser la demande.

Objectifs :
- Courbes élec par scénario (pointillés >2025), écarts et CAGR.
- Sankey du mix (Fossile / Bas-carbone / Renouvelable / Autres) en 2024/2035/2050.
- Part IA/crypto dans la demande mondiale (si pertinent).


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9,5)
root_pre = Path('..') / 'data' / 'preproc'


In [ ]:
elec = pd.read_excel(root_pre / 'world_total_electricity_generation.xlsx')
elec = elec.rename(columns={'Total_generation_TWh':'value','Scenario':'scenario','Year':'Year'})
elec['line_dash'] = np.where(elec['Year']>2025,'dash','solid')
fig = px.line(elec, x='Year', y='value', color='scenario', line_dash='line_dash', markers=True,
              title='Production/consommation électrique mondiale (pointillés >2025)')
fig.show()


In [ ]:
# Écarts 2035/2050
summary = elec[elec['Year'].isin([2035,2050])].pivot(index='Year', columns='scenario', values='value')
summary


In [ ]:
# CAGR 2024->2035/2050
cagr_rows=[]
for sc in elec['scenario'].unique():
    sc_data = elec[elec['scenario']==sc]
    if 2024 in sc_data['Year'].values:
        base = sc_data.loc[sc_data['Year']==2024,'value'].values[0]
        for target in [2035,2050]:
            if target in sc_data['Year'].values and base>0:
                val = sc_data.loc[sc_data['Year']==target,'value'].values[0]
                yrs = target-2024
                cagr_rows.append({'scenario':sc,'target_year':target,'CAGR':(val/base)**(1/yrs)-1})
pd.DataFrame(cagr_rows)


## Sankey du mix électrique


In [ ]:
gen = pd.read_excel(root_pre / 'total_power_generation.xlsx')
cat_map = {'Coal':'Fossile','Oil':'Fossile','Natural gas':'Fossile',
           'Nuclear':'Bas-carbone','Hydro':'Renouvelable','Solar PV':'Renouvelable','Wind':'Renouvelable'}
cols = [c for c in gen.columns if c!='Technology']
horizons=[]
for target in ['2024','2035','2050']:
    cand = [c for c in cols if target in str(c) and ('current' in str(c).lower())]
    if not cand:
        cand = [c for c in cols if target in str(c) and ('stated' in str(c).lower())]
    if not cand:
        cand = [c for c in cols if target in str(c)]
    if cand:
        horizons.append((target, cand[0]))

sankey_figs=[]
for target,col in horizons:
    mix = gen[['Technology', col]].dropna()
    mix['Category'] = mix['Technology'].map(cat_map).fillna('Autres')
    agg = mix.groupby('Category')[col].sum().reset_index()
    groups = agg['Category'].tolist() + [f"Electricité {target}"]
    idx = {n:i for i,n in enumerate(groups)}
    sources = [idx[c] for c in agg['Category']]
    targets = [idx[f"Electricité {target}"]]*len(agg)
    values = agg[col].tolist()
    fig = go.Figure(go.Sankey(node=dict(label=groups), link=dict(source=sources, target=targets, value=values)))
    fig.update_layout(title_text=f"Mix électrique {target} ({col})", font_size=12)
    sankey_figs.append(fig)

for f in sankey_figs:
    f.show()


## Part IA/crypto dans l’électricité (optionnel)


In [ ]:
try:
    dc = pd.read_excel(root_pre / 'data_center_electricity_demand_TWh.xlsx')
    merged = dc.merge(elec[['Year','value']].drop_duplicates(), on='Year', how='left')
    merged['Part_%'] = merged['Total']/merged['value']*100
    fig = px.line(merged, x='Year', y='Part_%', markers=True, title='Part IA/crypto dans la demande électrique mondiale')
    fig.show()
except FileNotFoundError:
    pass


## Lecture rapide
- Trajectoires élec par scénario : tension current vs stated/netzero (pointillés >2025).
- Écarts et CAGR : quantifient la montée en puissance nécessaire.
- Sankey : visualise la dépendance fossile résiduelle et la montée des renouvelables/bas-carbone.
- Part IA/crypto : illustre la pression additionnelle du numérique.
